# 00 · Data understanding and preprocessing
Runs on **CPU** (no GPU quota needed). Produces the statistics that fix later choices:
input resolution, anchor sizes, max detections / number of queries, frame step, handling of ignored regions.

## 0. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'  # <- change once: your public repo with src/common.py
FAST_DEV = os.environ.get('FAST_DEV') == '1'                # tiny smoke test on a fake dataset; keep off on Kaggle

if not any(Path(d, 'common.py').exists() for d in ['src', '../src', 'repo/src']):
    subprocess.run(['git', 'clone', '-q', REPO_URL, 'repo'], check=True)
sys.path.insert(0, next(d for d in ['src', '../src', 'repo/src'] if Path(d, 'common.py').exists()))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pycocotools'], check=True)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from common import *

seed_everything()
ROOT = find_data_root()
OUT = Path('results'); OUT.mkdir(exist_ok=True)
FIG = Path('images'); FIG.mkdir(exist_ok=True)  # figures for report.md
print('data root:', ROOT)

## 1. Load annotations (COCO json has track_id / occlusion, handy for EDA)

In [ ]:
def load_split(split):
    d = json.load(open(ROOT / split / 'annotations_coco.json'))
    imgs = pd.DataFrame(d['images']).rename(columns={'id': 'image_id'})
    anns = pd.DataFrame(d['annotations'])
    anns = anns.merge(imgs[['image_id', 'video', 'frame_id', 'width', 'height']], on='image_id')
    anns[['x', 'y', 'w', 'h']] = pd.DataFrame(anns.bbox.tolist(), index=anns.index)
    anns['size'] = np.sqrt(anns.w * anns.h)  # sqrt(area) in pixels
    anns['split'] = imgs['split'] = split
    anns['uid'] = split + '_' + anns.image_id.astype(str)  # image ids restart in every json
    imgs['uid'] = split + '_' + imgs.image_id.astype(str)
    return imgs, anns

imgs_tr, anns_tr = load_split('train')
imgs_va, anns_va = load_split('val')
imgs = pd.concat([imgs_tr, imgs_va], ignore_index=True); anns = pd.concat([anns_tr, anns_va], ignore_index=True)
objs = anns[anns.iscrowd == 0].copy()
objs['cls'] = objs.category_id.map(lambda c: CLASSES[c - 1])
ign = anns[anns.iscrowd == 1]
print(f'frames: {len(imgs)}  objects: {len(objs)}  ignored regions: {len(ign)}')
print('train/val clip overlap:', set(imgs_tr.video) & set(imgs_va.video) or 'none')
print('extra annotation fields:', [c for c in anns.columns if c not in ('bbox', 'segmentation')])

## 2. Clips and image sizes

In [ ]:
clips = imgs.groupby(['split', 'video']).agg(frames=('image_id', 'size'), width=('width', 'first'), height=('height', 'first')).reset_index()
print(clips.groupby('split').agg(clips=('video', 'size'), frames=('frames', 'sum')))
print('\nimage sizes (clips):'); print(clips.groupby(['width', 'height']).size().rename('clips'))
test = pd.read_csv(ROOT / 'test_images.csv')
print('\ntest image sizes (frames):'); print(test.groupby(['width', 'height']).size().rename('frames'))

## 3. Class distribution

In [ ]:
cls_tab = pd.crosstab(objs.cls, objs.split).reindex(CLASSES)
cls_tab['share_%'] = (100 * cls_tab.sum(1) / cls_tab.sum().sum()).round(1)
print(cls_tab)
ax = cls_tab[['train', 'val']].plot.bar(figsize=(8, 3.5), logy=True, title='Objects per class (log scale)')
plt.tight_layout(); plt.savefig(FIG / 'eda_classes.png', dpi=120); plt.show()

## 4. Object sizes
`size = sqrt(w*h)` in original pixels. COCO "small" = size < 32.

In [ ]:
print(objs['size'].describe(percentiles=[.1, .25, .5, .75, .9]).round(1))
print('share small (<32px): %.1f%%   tiny (<16px): %.1f%%' % (100 * (objs['size'] < 32).mean(), 100 * (objs['size'] < 16).mean()))
print('\nmedian size per class:'); print(objs.groupby('cls')['size'].median().reindex(CLASSES).round(1))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(objs['size'], bins=np.logspace(0, 2.7, 60)); ax[0].set_xscale('log'); ax[0].set_title('sqrt(area), px'); ax[0].axvline(32, c='r', ls='--')
objs.boxplot(column='size', by='cls', ax=ax[1], showfliers=False); ax[1].set_title('size by class'); plt.suptitle('')
plt.tight_layout(); plt.savefig(FIG / 'eda_sizes.png', dpi=120); plt.show()

**Resolution check.** Detectors resize the frame; the long side becomes `imgsz`. How big are objects after that?

In [ ]:
scale_rows = []
for long_side in [640, 1024, 1280, 1333, 1920]:
    s = objs['size'] * long_side / objs[['width', 'height']].max(1)
    scale_rows.append(dict(long_side=long_side, median_px=round(s.median(), 1),
                           pct_below_8px=round(100 * (s < 8).mean(), 1), pct_below_16px=round(100 * (s < 16).mean(), 1)))
print(pd.DataFrame(scale_rows).to_string(index=False))

## 5. Scene density (objects per frame)

In [ ]:
per_frame = objs.groupby('uid').size().reindex(imgs.uid, fill_value=0)
print(per_frame.describe(percentiles=[.5, .9, .99]).round(1))
print('frames with > 300 objects:', int((per_frame > 300).sum()), '| > 100:', int((per_frame > 100).sum()))
plt.figure(figsize=(6, 3)); plt.hist(per_frame, bins=60); plt.title('objects per frame'); plt.axvline(300, c='r', ls='--', label='300 (default DETR queries / YOLO max_det)'); plt.legend()
plt.tight_layout(); plt.savefig(FIG / 'eda_density.png', dpi=120); plt.show()

## 6. Variation across sequences

In [ ]:
per_clip = objs.groupby(['split', 'video']).agg(objects=('uid', 'size'), median_size=('size', 'median'))
per_clip['objs_per_frame'] = (per_clip.objects / clips.set_index(['split', 'video']).frames).round(1)
share = pd.crosstab([objs.split, objs.video], objs.cls, normalize='index').reindex(columns=CLASSES).round(2)
per_clip = per_clip.join(share)
print(per_clip.sort_values('objs_per_frame', ascending=False).head(10))
print('\nspread across clips (std / mean):')
print((per_clip[['objs_per_frame', 'median_size']].std() / per_clip[['objs_per_frame', 'median_size']].mean()).round(2))

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(share.T.values, aspect='auto', cmap='viridis'); ax.set_yticks(range(NC)); ax.set_yticklabels(CLASSES)
ax.set_xlabel('clip'); ax.set_title('class share per clip'); plt.colorbar(im)
plt.tight_layout(); plt.savefig(FIG / 'eda_clip_classes.png', dpi=120); plt.show()

## 7. Ignored regions and occlusion

In [ ]:
frames_with_ign = ign.uid.nunique()
ign_area = (ign.w * ign.h / (ign.width * ign.height)).groupby(ign.uid).sum()
print(f'frames with ignored regions: {frames_with_ign} / {len(imgs)} ({100 * frames_with_ign / len(imgs):.1f}%)')
print(f'mean share of frame area ignored (when present): {100 * ign_area.mean():.1f}%')
if 'occlusion' in objs:
    print('\nocclusion level share (0=none, 1=partial, 2=heavy):'); print(objs.occlusion.value_counts(normalize=True).sort_index().round(3))

## 8. Temporal redundancy → frame step
Share of objects (track_id) from frame *t* still present at *t+k*, and IoU of the same object between the two frames.
If frames *t* and *t+k* are almost the same, training on both adds little.

In [ ]:
def track_overlap(df, k):
    a = df[['video', 'frame_id', 'track_id', 'x', 'y', 'w', 'h']]
    b = a.assign(frame_id=a.frame_id - k)
    m = a.merge(b, on=['video', 'frame_id', 'track_id'], suffixes=('', '_n'))
    x1 = np.maximum(m.x, m.x_n); y1 = np.maximum(m.y, m.y_n)
    x2 = np.minimum(m.x + m.w, m.x_n + m.w_n); y2 = np.minimum(m.y + m.h, m.y_n + m.h_n)
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    iou = inter / (m.w * m.h + m.w_n * m.h_n - inter)
    return len(m) / len(a), iou.median()

if 'track_id' in objs:
    red = pd.DataFrame([(k, *track_overlap(objs[objs.split == 'train'], k)) for k in [1, 2, 5, 10, 20]],
                       columns=['step', 'objects_still_present', 'median_IoU_same_object']).round(3)
    print(red.to_string(index=False))

## 9. Anchor statistics (k-means on box shapes, as in YOLOv2)
Boxes are rescaled to the torchvision input scale (long side 1333) used by Faster R-CNN / RetinaNet.

In [ ]:
from sklearn.cluster import KMeans
scale = 1333 / objs[['width', 'height']].max(1)
wh = np.stack([objs.w * scale, objs.h * scale], 1)
sample = wh[np.random.RandomState(SEED).choice(len(wh), min(200000, len(wh)), replace=False)]
km = KMeans(9, n_init=5, random_state=SEED).fit(np.log(sample))  # log-space k-means: balances small and large boxes
centers = np.exp(km.cluster_centers_)
centers = centers[np.argsort(centers.prod(1))]
anchor_tab = pd.DataFrame({'w': centers[:, 0], 'h': centers[:, 1], 'size=sqrt(wh)': np.sqrt(centers.prod(1)),
                           'ratio h/w': centers[:, 1] / centers[:, 0]}).round(1)
print(anchor_tab.to_string(index=False))
print('\nh/w quantiles (25/50/75%):', np.quantile(wh[:, 1] / wh[:, 0], [.25, .5, .75]).round(2))
print('size quantiles at 1333 (5/50/95%):', np.quantile(np.sqrt(wh.prod(1)), [.05, .5, .95]).round(1))
anchor_tab.to_csv(OUT / 'eda_kmeans_anchors.csv', index=False)

## 10. Verify label conversion: YOLO .txt boxes drawn on frames
Green = YOLO label file, grey hatch = ignored region. Boxes must sit on the objects.

In [ ]:
vf = list_frames(ROOT / 'val', 1)
sample_frames = [vf[i] for i in np.linspace(0, len(vf) - 1, 3).astype(int)]
fig, axes = plt.subplots(len(sample_frames), 1, figsize=(14, 7 * len(sample_frames)))
for ax, (c, p) in zip(np.atleast_1d(axes), sample_frames):
    show_frame(ROOT, p, c, None, ax=ax, title=f'{c}/{p.name}')
plt.tight_layout(); plt.savefig(FIG / 'eda_gt_boxes.png', dpi=100); plt.show()

## 11. Masking ignored regions — check on one frame
Ignored areas are filled with grey (pixels of labelled objects restored) so the detector is not trained to call unlabelled crowds "background".

In [ ]:
c, p = next((c, p) for c, p in list_frames(ROOT / 'train', 1) if ignore_path_for(ROOT, 'train', c, p).exists())
img = cv2.imread(str(p)); h, w = img.shape[:2]
gt, _ = read_yolo_labels(label_path_for(ROOT, 'train', c, p), w, h)
masked = mask_ignored(img, read_ignore(ignore_path_for(ROOT, 'train', c, p), w, h), gt)
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
ax[0].imshow(img[:, :, ::-1]); ax[0].set_title('original'); ax[1].imshow(masked[:, :, ::-1]); ax[1].set_title('ignored regions masked')
[a.axis('off') for a in ax]; plt.tight_layout(); plt.savefig(FIG / 'eda_masking.png', dpi=100); plt.show()

## 12. Validation design
* The provided `val/` split = 7 whole clips, no clip shared with `train/` (checked in section 1) → sequence-level hold-out, no frame leakage.
* All detectors are compared on the **same** val frames (every 5th frame of the 7 clips) with the **same** COCO evaluator
  (`common.CocoEvaluator`: ignored regions as `iscrowd`, maxDets = 500 like the leaderboard).
* Sequence-level folds over the training clips (GroupKFold) are saved below for optional extra folds when GPU budget allows.

In [ ]:
from sklearn.model_selection import GroupKFold
tr_clips = clips[clips.split == 'train'].reset_index(drop=True)
gkf = GroupKFold(n_splits=min(3, len(tr_clips)))
tr_clips['fold'] = -1
for f, (_, idx) in enumerate(gkf.split(tr_clips, groups=tr_clips.video)):
    tr_clips.loc[idx, 'fold'] = f
tr_clips[['video', 'frames', 'fold']].to_csv(OUT / 'folds.csv', index=False)
print(tr_clips.groupby('fold').agg(clips=('video', 'size'), frames=('frames', 'sum')))

## 13. Observations → decisions
*(write your own conclusions here, based on the numbers above)*
* Resolution: …
* Anchors (RetinaNet / Faster R-CNN): …
* Max detections / number of DETR queries: …
* Frame step: …
* Class imbalance: …